# Plot results for offline prompting in the "dragon" example

In [1]:
%cd ..
%pwd  # should be "llm-adaptation"

C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation


C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation\.venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


'C:\\Users\\micha\\OneDrive - Univerzita Karlova\\research\\2024-LLM-DEECo\\llm-adaptation'

In [2]:
from pathlib import Path
import pandas as pd
import shutil
import subprocess
import sys
import json

In [30]:
results_folder = Path("generated_adaptations/dragon")
pd.options.display.float_format = "{:,.1f}".format

## Run experiments

In [34]:
prompt = "generated_adaptations/prompts/dragon_strategy.md"
variants = {
    "default": [],
    "notest": ["--retries_test=0"],
    "constraints": []
}
llms = {
    # "5nanolow": "gpt-5-nano-2025-08-07,reasoning_effort=low",
    # "5nano": "gpt-5-nano-2025-08-07",
    "5mini": "gpt-5-mini-2025-08-07",
    # "5": "gpt-5-2025-08-07",
}
repeats = 3
start = 1

In [35]:
# import generated_adaptations.generator as generator

In [36]:
for variant, args in variants.items():
    for llm_name, llm in llms.items():
        for repeat in range(start, start + repeats):
            folder_name = f"{llm_name}_{repeat:02d}"
            print(f"\n{variant}/{folder_name}\n")
            folder = results_folder / variant / folder_name
            folder.mkdir(parents=True, exist_ok=True)
            shutil.copy(prompt, folder / "01_01_user.md")
            cmd  = [sys.executable, "generated_adaptations/generator.py", f"--folder={str(folder)}", f"--llm={llm}", *args]
            result = subprocess.run(cmd, capture_output=True, text=True)  # even capture_output=False does not stream in real-time, so we rather capture it and print it later
            print(result.stdout)
            print(result.stderr)
            # generator.main([f"--folder={str(folder)}"])  # this also does not show the real-time output


default/5mini_01

Loaded 2 messages from generated_adaptations\dragon\default\5mini_01.
Querying LLM with prompt 01_01_user.
LLM response saved to '01_02_llm.md'.
Code block saved to 'generated_adaptations\dragon\default\5mini_01\code_01_02.py'.
TOKENS USED:
Input: 1066, Output: 3721 (reasoning: 1984)
Response time (seconds): 57.2

Running tests: pytest generated_adaptations/tests -q --tb=short -rfExXpP --show-capture=no --color=no --example=dragon --adaptation_name=5mini_01/code_01_02 --variant=default
Test exit code: 0
Running simulation for 'generated_adaptations\dragon\default\5mini_01\code_01_02.py'.
  Run #1/3: python main.py dragon/configs/default.yaml generated_adaptations/configs/generated.yaml DSL/dragon.yaml --extra_config={"name": "default/5mini_01/code_01_02", "log_dir.append": "/default/5mini_01/code_01_02", "adaptation_name": "generated_adaptations.dragon.default.5mini_01.code_01_02.SmartAdaptation"} -s 1 -e 1
  Run #2/3: python main.py dragon/configs/default.yaml gener

## Results

In [37]:
folders = list(results_folder.glob("*/*"))
# folders = list(results_folder.glob("*/5nano_*"))
# folders = list(results_folder.glob("*/5nanolow*"))
# folders = [results_folder / "41mini"]
print([(f.parent.stem, f.stem) for f in folders])

[('constraints', '5mini_01'), ('constraints', '5mini_02'), ('constraints', '5mini_03'), ('constraints', '5nanolow_01'), ('constraints', '5nanolow_02'), ('constraints', '5nanolow_03'), ('constraints', '5nano_01'), ('constraints', '5nano_02'), ('constraints', '5nano_03'), ('default', '5mini_01'), ('default', '5mini_02'), ('default', '5mini_03'), ('default', '5nanolow_01'), ('default', '5nanolow_02'), ('default', '5nanolow_03'), ('default', '5nano_01'), ('default', '5nano_02'), ('default', '5nano_03'), ('notest', '5mini_01'), ('notest', '5mini_02'), ('notest', '5mini_03'), ('notest', '5nanolow_01'), ('notest', '5nanolow_02'), ('notest', '5nanolow_03'), ('notest', '5nano_01'), ('notest', '5nano_02'), ('notest', '5nano_03')]


In [38]:
summary = pd.DataFrame(columns=["llm", "params", "repeat"])
best = pd.DataFrame(columns=["llm", "params", "repeat"])
for folder in folders:
    llm, repeat = folder.stem.split("_")
    params = folder.parent.stem
    summary.loc[len(summary), ["llm", "params", "repeat"]] = [llm, params, repeat]
    best.loc[len(best), ["llm", "params", "repeat"]] = [llm, params, repeat]
    for file in (folder / "results").glob("*.txt"):
        name = file.stem.removeprefix("code_")
        if "_test_fail" in name:
            code = name.removesuffix("_test_fail")
            summary.loc[len(summary) - 1, code + "_test"] = "fail"
            best.loc[len(best) - 1, code.split("_")[0] + "_test"] = "fail"
        elif "_test_pass" in name:
            code = name.removesuffix("_test_pass")
            summary.loc[len(summary) - 1, code + "_test"] = "pass"
            best.loc[len(best) - 1, code.split("_")[0] + "_test"] = "pass"
        else:
            print(f"Unknown file: {file}")
    for file in (folder / "results").glob("*.json"):
        name = file.stem.removeprefix("code_")
        if "_simulation_result" in name:
            code = name.removesuffix("_simulation_result")
            results = json.load(open(file))
            # result = results["winrate"]
            result = results["steps"]
            summary.loc[len(summary) - 1, code + "_result"] = result
            best.loc[len(best) - 1, code.split("_")[0] + "_result"] = result  # only take the first part of the code file name
        else:
            print(f"Unknown file: {file}")
summary = summary.astype("object")
best = best.astype("object")
summary.fillna("-", inplace=True)
best.fillna("-", inplace=True)

In [39]:
summary.set_index(["llm", "params", "repeat"], inplace=True)
best.set_index(["llm", "params", "repeat"], inplace=True)

In [40]:
summary = summary.reindex(sorted(summary.columns), axis=1)
best = best.reindex(sorted(best.columns), axis=1)

In [41]:
summary

01_02_result 01_02_test 01_04_result 01_04_test  \
llm      params      repeat                                                   
5mini    constraints 01                -       fail         12.0       pass   
                     02             12.3       pass            -          -   
                     03                -       fail            -       fail   
5nanolow constraints 01                -       fail            -       fail   
                     02                -       fail            -       fail   
                     03                -       fail            -       fail   
5nano    constraints 01                -       fail         12.0       pass   
                     02             12.0       pass            -          -   
                     03                -       fail            -       fail   
5mini    default     01                -       pass            -          -   
                     02                -       pass            -          -   
                     03                -       pass            -          -   
5nanolow default     01                -       fail            -       pass   
                     02                -       fail            -       fail   
                     03                -       fail            -       fail   
5nano    default     01                -       pass            -          -   
                     02                -       pass            -          -   
                     03                -       pass            -          -   
5mini    notest      01                -       pass            -          -   
                     02                -       pass            -          -   
                     03                -       pass            -          -   
5nanolow notest      01                -       pass            -          -   
                     02                -       fail            -          -   
                     03                -       fail            -          -   
5nano    notest      01                -       fail            -          -   
                     02                -       pass            -          -   
                     03                -       pass            -          -   

                            01_06_result 01_06_test 01_08_result 01_08_test  \
llm      params      repeat                                                   
5mini    constraints 01                -          -            -          -   
                     02                -          -            -          -   
                     03                -       fail         12.0       pass   
5nanolow constraints 01                -       fail            -       fail   
                     02             12.0       pass            -          -   
                     03                -       fail            -       fail   
5nano    constraints 01                -          -            -          -   
                     02                -          -            -          -   
                     03             12.0       pass            -          -   
5mini    default     01                -          -            -          -   
                     02                -          -            -          -   
                     03                -          -            -          -   
5nanolow default     01                -          -            -          -   
                     02                -       pass            -          -   
                     03                -       fail            -       pass   
5nano    default     01                -          -            -          -   
                     02                -          -            -          -   
                     03                -          -            -          -   
5mini    notest      01                -          -            -          -   
                     02                -          -            -          -  

In [42]:
best

01_result 01_test 02_result 02_test 03_result  \
llm      params      repeat                                                 
5mini    constraints 01          12.0    pass         -    fail      15.7   
                     02          12.3    pass         -    fail      12.0   
                     03          12.0    pass      12.0    pass         -   
5nanolow constraints 01             -    fail         -    fail         -   
                     02          12.0    pass      12.0    pass      12.0   
                     03             -    fail         -    fail         -   
5nano    constraints 01          12.0    pass      12.0    pass      12.0   
                     02          12.0    pass      12.0    pass      15.7   
                     03          12.0    pass      12.0    pass      12.0   
5mini    default     01             -    pass      12.0    pass      12.0   
                     02             -    pass      12.3    pass      12.3   
                     03             -    pass      12.0    pass         -   
5nanolow default     01             -    pass         -    pass         -   
                     02             -    pass         -    pass         -   
                     03             -    pass         -    pass         -   
5nano    default     01             -    pass      16.3    pass      12.3   
                     02             -    pass         -    pass      12.0   
                     03             -    pass      12.0    pass      12.0   
5mini    notest      01             -    pass         -    pass         -   
                     02             -    pass      12.0    pass      12.0   
                     03             -    pass      12.0    pass      20.3   
5nanolow notest      01             -    pass      12.0    pass      12.0   
                     02             -    fail         -    pass         -   
                     03             -    fail         -    fail         -   
5nano    notest      01             -    fail         -    pass         -   
                     02             -    pass      12.0    pass      12.7   
                     03             -    pass         -    pass         -   

                            03_test  
llm      params      repeat          
5mini    constraints 01        pass  
                     02        pass  
                     03           -  
5nanolow constraints 01        fail  
                     02        pass  
                     03        fail  
5nano    constraints 01        pass  
                     02        pass  
                     03        pass  
5mini    default     01        pass  
                     02        pass  
                     03        pass  
5nanolow default     01        pass  
                     02        pass  
                     03        pass  
5nano    default     01        pass  
                     02        pass  
                     03        pass  
5mini    notest      01        pass  
                     02        pass  
                     03        pass  
5nanolow notest      01        pass  
                     02        fail  
                     03        fail  
5nano    notest      01        pass  
                     02        pass  
                     03        pass